# 09 Report Builder

作るもの: `predictions_v1` / `metrics_v1` / `clips_v1` / CV artifacts から static HTML report と full pipeline dashboard を生成します。

必要な入力: `predictions/{RUN_ID}/predictions_v1.*`, `predictions/{RUN_ID}/metrics_v1.json`

任意入力: `clips/{FULL_RUN_ID}/clips_v1.parquet`, `debug/{FULL_RUN_ID}/cv_overlay_videos_v1.parquet`, `predictions/{RUN_ID}/fusion_input_audit_v1.parquet`

保存先: `/content/drive/MyDrive/baseball_vision/reports/{report_name}/{RUN_ID}/index.html`

作成される report: pipeline dashboard, target availability, experiment compare, failure browser, clip quality

無いとまずいもの: 05b/13/18/19/15 の少なくともどれかの `predictions_v1.*` と `metrics_v1.json`。xBA / xwOBA missing と OPS unavailable は report に表示されます。

この notebook は heavy training / video download / model download を実行しません。


In [ ]:
from pathlib import Path
import os
import sys
import importlib.util


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    try:
        drive.mount(str(mountpoint))
    except ValueError as exc:
        message = str(exc)
        if 'Mountpoint must not already contain files' in message or 'already mounted' in message:
            print(f'Drive mount warning: {message}')
            if not (mountpoint / 'MyDrive').exists():
                drive.mount(str(mountpoint), force_remount=True)
        else:
            raise
    except Exception as exc:
        print(f'Colab Drive mount skipped or failed: {exc}')


def _is_repo_dir(path: Path) -> bool:
    return (
        (path / 'src' / 'sport_pipeline' / '__init__.py').exists()
        and (path / 'configs').exists()
        and (path / 'notebooks').exists()
    )


def _resolve_repo_dir() -> Path:
    fixed = Path('/content/drive/MyDrive/codex/batting_codex_handoff')
    candidates: list[Path] = []
    env_root = os.environ.get('BATTING_CODE_ROOT')
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend(
        [
            fixed,
            Path.cwd(),
        ]
    )
    candidates.extend(Path.cwd().parents)

    checked: list[str] = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            if candidate != fixed:
                print('WARNING: fixed REPO_DIR ではない場所の repo を使います。')
                print('  fixed:', fixed)
                print('  using:', candidate)
                print('  次回は repo フォルダを fixed path に置くか、BATTING_CODE_ROOT を明示してください。')
            return candidate

    raise FileNotFoundError(
        'sport_pipeline package が見つかりません。Drive には notebook 単体ではなく repo フォルダごと置く必要があります。\n'
        '推奨配置: /content/drive/MyDrive/codex/batting_codex_handoff\n'
        '別の場所に置いた場合は、この notebook の最初の cell より前に次を実行してください。\n'
        '  %env BATTING_CODE_ROOT=/content/drive/MyDrive/batting_codex_handoff\n'
        '確認した候補:\n- ' + '\n- '.join(checked)
    )


_mount_drive_if_needed()

REPO_DIR = _resolve_repo_dir()
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ.get('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME

BASE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)

src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(
        'sport_pipeline を import できません。repo 配置と src/sport_pipeline/__init__.py を確認してください。\n'
        f'REPO_DIR={REPO_DIR}\n'
        f'src_dir={src_dir}\n'
        f'src_dir_exists={src_dir.exists()}'
    )

print('REPO_DIR =', REPO_DIR)
print('BASE_DIR =', BASE_DIR)
print('CACHE_DIR =', CACHE_DIR)
print('RUN_PROFILE_PATH =', RUN_PROFILE_PATH)
print('src_dir =', src_dir)

from sport_pipeline.pipeline.run_profile import load_run_profile, run_id
from sport_pipeline.reports.run_selection import select_report_run_id

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
# Set this to a concrete run id when you want a specific report, e.g. fusion/context/sequence/video.
PREFERRED_RUN_ID = None
selection = select_report_run_id(BASE_DIR, RUN_PROFILE, preferred_run_id=PREFERRED_RUN_ID, include_smoke=False)
RUN_ID = selection.run_id
FULL_RUN_ID = run_id(RUN_PROFILE, 'full_run_id')

print('RUN_ID =', RUN_ID)
print('FULL_RUN_ID =', FULL_RUN_ID)
print('report_run_selection =', selection)
if not selection.available:
    print('No reportable prediction run was found yet. Missing required inputs for selected fallback:')
    for missing_path in selection.missing_required:
        print('missing:', missing_path)


In [ ]:
from sport_pipeline.artifact_check import check_stage_artifacts
from sport_pipeline.reports.build_static import (
    build_static_reports,
    read_metrics_payload,
    read_table_artifact,
)
from sport_pipeline.reports.pipeline_dashboard import (
    available_metrics_payloads,
    build_pipeline_dashboard,
)
from sport_pipeline.reports.run_selection import metrics_artifact_path, prediction_artifact_path

predictions_path = prediction_artifact_path(BASE_DIR, RUN_ID)
metrics_path = metrics_artifact_path(BASE_DIR, RUN_ID)
clips_path = BASE_DIR / 'clips' / FULL_RUN_ID / 'clips_v1.parquet'
fusion_audit_path = BASE_DIR / 'predictions' / RUN_ID / 'fusion_input_audit_v1.parquet'
overlay_manifest_path = BASE_DIR / 'debug' / FULL_RUN_ID / 'cv_overlay_videos_v1.parquet'

required = [predictions_path, metrics_path]
missing_required = [path for path in required if not path.exists()]
if missing_required:
    print('必要な artifact がありません。RUN_ID に対応する predictions/metrics を先に作成してください。')
    for path in missing_required:
        print('missing:', path)
    raise FileNotFoundError('report input artifact missing')

optional_clip_rows = []
if clips_path.exists():
    optional_clip_rows = read_table_artifact(clips_path)
else:
    print('optional FULL_RUN clips_v1 not found; clip quality report will be generated with empty clip rows:', clips_path)

if overlay_manifest_path.exists():
    overlay_rows = read_table_artifact(overlay_manifest_path)
    overlay_by_clip = {str(row.get('clip_id')): row for row in overlay_rows if row.get('clip_id') is not None}
    for clip in optional_clip_rows:
        clip_id = str(clip.get('clip_id'))
        overlay = overlay_by_clip.get(clip_id)
        if overlay and not clip.get('overlay_path'):
            clip['overlay_path'] = overlay.get('artifact_path')
    print('overlay manifest rows =', len(overlay_rows), overlay_manifest_path)
else:
    print('optional overlay manifest not found; run 21_cv_overlay_videos.ipynb for YOLO/pose overlay mp4 links:', overlay_manifest_path)

optional_fusion_audit_rows = []
if fusion_audit_path.exists():
    optional_fusion_audit_rows = read_table_artifact(fusion_audit_path)
else:
    print('optional fusion_input_audit_v1 not found; fusion audit section will use prediction metadata only:', fusion_audit_path)

prediction_rows = read_table_artifact(predictions_path)
metrics_payload = read_metrics_payload(metrics_path)
all_metrics_payloads = available_metrics_payloads(BASE_DIR, RUN_PROFILE)
if not all_metrics_payloads:
    all_metrics_payloads = [metrics_payload]

dashboard_paths = build_pipeline_dashboard(
    base_dir=BASE_DIR,
    run_profile=RUN_PROFILE,
    dashboard_id=RUN_ID,
)

paths = build_static_reports(
    base_dir=BASE_DIR,
    run_id=RUN_ID,
    prediction_rows=prediction_rows,
    metrics_payloads=all_metrics_payloads,
    clip_rows=optional_clip_rows,
    fusion_audit_rows=optional_fusion_audit_rows,
)
paths.update(dashboard_paths)

for name, path in paths.items():
    print(name, '->', path, 'exists=', Path(path).exists())

check = check_stage_artifacts('reports', base_dir=BASE_DIR, run_id=RUN_ID)
print(check)
if not check['all_present']:
    raise RuntimeError('report artifact が不足しています。上の出力 path と RUN_ID を確認してください。')


In [ ]:
from IPython.display import IFrame, display

report_path = BASE_DIR / 'reports' / 'pipeline_dashboard' / RUN_ID / 'index.html'
print('Open this full pipeline dashboard from Google Drive:')
print(report_path)
print('Experiment compare:', BASE_DIR / 'reports' / 'experiment_compare' / RUN_ID / 'index.html')
print('Clip quality:', BASE_DIR / 'reports' / 'clip_quality' / RUN_ID / 'index.html')
print('Failure browser:', BASE_DIR / 'reports' / 'failure_browser' / RUN_ID / 'index.html')

if report_path.exists():
    display(IFrame(str(report_path), width='100%', height=800))
else:
    print('report がまだ生成されていません。前の cell を実行してください。')


## Optional Smoke Report

real artifact が無い状態で report builder 自体だけを確認したい場合は、下の `RUN_SMOKE` を `True` にして実行します。通常は実行不要です。


In [ ]:
RUN_SMOKE = False

if RUN_SMOKE:
    from sport_pipeline.reports.build_static import run_report_smoke

    smoke_run_id = 'phase9_smoke'
    smoke_paths = run_report_smoke(BASE_DIR, smoke_run_id)
    for name, path in smoke_paths.items():
        print(name, '->', path, 'exists=', Path(path).exists())
else:
    print('Smoke report は skip しました。real artifact report が通常の確認対象です。')
